# SIRCBF — Avance 2 (versión corregida)
## Sistema de reconocimiento y clasificación de especímenes de billetes mediante K-NN

Este cuaderno descarga el conjunto oficial UCI Banknote Authentication, revisa los datos, genera el análisis exploratorio, calcula un baseline y ejecuta una implementación propia de K-NN.

**Corrección metodológica:** las observaciones con las mismas cuatro características se mantienen juntas en un único subconjunto. Así evitamos que una copia aparezca en entrenamiento y otra en validación o prueba. El conjunto de prueba permanece reservado. Las etiquetas se mantienen como Clase 0 y Clase 1.

In [ ]:
!pip -q install numpy pandas matplotlib ucimlrepo
print('Dependencias instaladas. Continúa con la siguiente celda.')

## Ejecución del análisis y primera versión del núcleo

Ejecuta la siguiente celda. Además de las gráficas y métricas, el programa informa si existen grupos de características repetidas y verifica que ningún grupo pase a más de un subconjunto. Si encuentra la misma combinación de características con etiquetas diferentes, se detiene para que esos casos sean revisados.

In [ ]:
"""SIRCBF - Avance 2 (version corregida)
Clasificador K-NN implementado desde cero para UCI Banknote Authentication.

Correcciones metodologicas principales:
- La division se realiza por grupos de caracteristicas identicas: una misma
  combinacion de variance, skewness, curtosis y entropy nunca aparece en mas
  de un subconjunto.
- Se comprueba expresamente el solapamiento entre entrenamiento, validacion
  y prueba.
- La normalizacion se ajusta solo con entrenamiento.
- El conjunto de prueba se mantiene reservado en este avance.
- Si una misma combinacion de caracteristicas aparece con etiquetas distintas,
  el programa se detiene para revisar esa inconsistencia en vez de ocultarla.

Instalacion:
    pip install numpy pandas matplotlib ucimlrepo
Ejecucion:
    python avance2_sircbf.py
"""

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
FEATURES = ["variance", "skewness", "curtosis", "entropy"]
K_CANDIDATES = [1, 3, 5, 7, 9, 11]
OUT_DIR = Path("salidas_sircbf")


def load_uci_banknote():
    """Descarga el dataset oficial UCI (ID 267) y devuelve un DataFrame."""
    try:
        from ucimlrepo import fetch_ucirepo
    except ImportError as exc:
        raise SystemExit(
            "Falta ucimlrepo. Instala dependencias con: "
            "pip install numpy pandas matplotlib ucimlrepo"
        ) from exc

    dataset = fetch_ucirepo(id=267)
    X = dataset.data.features.copy()
    y = dataset.data.targets.copy()
    if y.shape[1] != 1:
        raise ValueError("Se esperaba una sola columna objetivo en el dataset UCI.")

    X.columns = [str(c).strip().lower() for c in X.columns]
    missing_columns = [col for col in FEATURES if col not in X.columns]
    if missing_columns:
        raise ValueError(f"Faltan columnas predictoras en UCI: {missing_columns}")
    X = X[FEATURES]
    y = y.iloc[:, 0].rename("class")
    return pd.concat([X, y], axis=1)


def clean_and_report(df):
    """Convierte columnas, informa faltantes/infinitos/duplicados y conserva filas."""
    df = df.copy()
    for col in FEATURES + ["class"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Contar faltantes antes de retirar cualquier fila.
    n_missing_before = int(df[FEATURES + ["class"]].isna().sum().sum())

    # Detectar infinitos solo en las variables predictoras.
    X_numeric = df[FEATURES].to_numpy(dtype=float)
    infinite_mask = np.isinf(X_numeric).any(axis=1)
    n_infinite = int(infinite_mask.sum())
    if n_infinite:
        df = df.loc[~infinite_mask].copy()

    n_missing_rows_removed = int(df[FEATURES + ["class"]].isna().any(axis=1).sum())
    df = df.dropna(subset=FEATURES + ["class"]).copy()

    # Reportar duplicados exactos y duplicados de caracteristicas.
    n_duplicate_full_rows = int(df.duplicated(subset=FEATURES + ["class"]).sum())
    n_duplicate_feature_rows = int(df.duplicated(subset=FEATURES).sum())

    if not np.all(np.isclose(df["class"].to_numpy(dtype=float) % 1, 0)):
        raise ValueError("La variable objetivo contiene etiquetas no enteras.")
    df["class"] = df["class"].astype(int)

    classes = set(df["class"].unique().tolist())
    if classes != {0, 1}:
        raise ValueError(f"Se esperaban las etiquetas binarias 0 y 1; se encontraron: {sorted(classes)}")

    print("\n=== CONTROL DE CALIDAD ===")
    print(f"Filas despues de retirar infinitos/faltantes: {len(df)}")
    print(f"Valores faltantes detectados antes de eliminarlos: {n_missing_before}")
    print(f"Filas con infinitos detectadas y retiradas: {n_infinite}")
    print(f"Filas retiradas por valores faltantes: {n_missing_rows_removed}")
    print(f"Filas duplicadas exactas (se conservan): {n_duplicate_full_rows}")
    print(f"Filas con caracteristicas repetidas (se conservan y agrupan): {n_duplicate_feature_rows}")
    print("\nValores faltantes por columna despues de limpieza:")
    print(df[FEATURES + ["class"]].isna().sum().to_string())
    return df.reset_index(drop=True)


def create_eda(df):
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    print("\n=== DIMENSIONES DEL DATASET ===")
    print(f"Muestras: {len(df)}")
    print(f"Variables predictoras: {len(FEATURES)}")
    print(f"Clases: {sorted(df['class'].unique().tolist())}")

    print("\n=== ESTADISTICAS DESCRIPTIVAS ===")
    print(df[FEATURES].describe().round(4).to_string())

    print("\n=== BALANCE DE CLASES ===")
    counts = df["class"].value_counts().sort_index()
    print(counts.to_string())
    print("Porcentajes:")
    print((100 * counts / counts.sum()).round(2).astype(str).add("%").to_string())

    print("\n=== CORRELACIONES ENTRE VARIABLES ===")
    corr = df[FEATURES].corr(numeric_only=True)
    print(corr.round(3).to_string())

    fig, axes = plt.subplots(2, 2, figsize=(11, 7))
    for ax, feature in zip(axes.flat, FEATURES):
        ax.hist(df[feature], bins=30, edgecolor="black")
        ax.set_title(f"Distribucion: {feature}")
        ax.set_xlabel(feature)
        ax.set_ylabel("Frecuencia")
    fig.tight_layout()
    fig.savefig(OUT_DIR / "01_distribuciones.png", dpi=160)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar([str(x) for x in counts.index], counts.values)
    ax.set_title("Balance de clases")
    ax.set_xlabel("Etiqueta de clase")
    ax.set_ylabel("Cantidad de muestras")
    fig.tight_layout()
    fig.savefig(OUT_DIR / "02_balance_clases.png", dpi=160)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(7, 6))
    image = ax.imshow(corr.to_numpy(), aspect="auto", vmin=-1, vmax=1)
    ax.set_xticks(range(len(FEATURES)), FEATURES, rotation=35, ha="right")
    ax.set_yticks(range(len(FEATURES)), FEATURES)
    ax.set_title("Correlacion entre variables")
    for i in range(len(FEATURES)):
        for j in range(len(FEATURES)):
            ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center")
    fig.colorbar(image, ax=ax, label="Correlacion")
    fig.tight_layout()
    fig.savefig(OUT_DIR / "03_correlaciones.png", dpi=160)
    plt.close(fig)
    print(f"\nGraficas guardadas en: {OUT_DIR.resolve()}")


def stratified_group_split_indices(X, y, train_ratio=0.70, val_ratio=0.15, seed=SEED):
    """Divide por grupos de filas con caracteristicas identicas y estratifica por clase.

    Todas las filas con la misma combinacion de caracteristicas van al mismo
    subconjunto. Esto evita que una copia de un registro aparezca en train y
    otra copia en validacion/prueba. Se conservan todas las filas originales.
    """
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=int)
    test_ratio = 1.0 - train_ratio - val_ratio
    if train_ratio <= 0 or val_ratio <= 0 or test_ratio <= 0:
        raise ValueError("Las proporciones deben ser positivas y sumar menos de 1 para train+val.")

    unique_X, inverse = np.unique(X, axis=0, return_inverse=True)
    n_groups = len(unique_X)
    group_labels = np.empty(n_groups, dtype=int)
    group_sizes = np.bincount(inverse, minlength=n_groups)
    conflicting_groups = []

    for group_id in range(n_groups):
        labels = np.unique(y[inverse == group_id])
        if len(labels) != 1:
            conflicting_groups.append((group_id, labels.tolist(), int(group_sizes[group_id])))
        else:
            group_labels[group_id] = int(labels[0])

    if conflicting_groups:
        preview = conflicting_groups[:10]
        raise ValueError(
            "Se encontraron combinaciones de caracteristicas identicas con etiquetas distintas. "
            f"No se puede hacer una division agrupada segura sin revisar esos casos. Muestra: {preview}"
        )

    rng = np.random.default_rng(seed)
    ratios = np.array([train_ratio, val_ratio, test_ratio], dtype=float)
    group_assignment = np.full(n_groups, -1, dtype=int)  # 0=train, 1=val, 2=test

    for label in np.unique(y):
        label_groups = np.flatnonzero(group_labels == label)
        if len(label_groups) < 3:
            raise ValueError(f"La clase {label} tiene menos de 3 grupos unicos; no se puede dividir en 3 partes.")

        # Aleatorizar empates y colocar primero los grupos mas grandes para
        # aproximar las proporciones sin separar sus filas repetidas.
        shuffled = rng.permutation(label_groups)
        order = sorted(shuffled.tolist(), key=lambda gid: -int(group_sizes[gid]))
        total_rows = int(group_sizes[label_groups].sum())
        targets = ratios * total_rows
        current = np.zeros(3, dtype=float)

        for group_id in order:
            size = float(group_sizes[group_id])
            candidate_scores = []
            for split_id in range(3):
                proposed = current.copy()
                proposed[split_id] += size
                score = np.sum(((proposed - targets) / np.maximum(targets, 1.0)) ** 2)
                candidate_scores.append(score)
            best_score = min(candidate_scores)
            best_splits = [i for i, score in enumerate(candidate_scores) if np.isclose(score, best_score)]
            selected_split = int(rng.choice(best_splits))
            group_assignment[group_id] = selected_split
            current[selected_split] += size

    train_idx = np.flatnonzero(group_assignment[inverse] == 0)
    val_idx = np.flatnonzero(group_assignment[inverse] == 1)
    test_idx = np.flatnonzero(group_assignment[inverse] == 2)
    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    rng.shuffle(test_idx)

    # Verificacion explicita: ningun vector de caracteristicas se repite entre partes.
    train_groups = set(inverse[train_idx].tolist())
    val_groups = set(inverse[val_idx].tolist())
    test_groups = set(inverse[test_idx].tolist())
    overlap_train_val = len(train_groups & val_groups)
    overlap_train_test = len(train_groups & test_groups)
    overlap_val_test = len(val_groups & test_groups)
    if overlap_train_val or overlap_train_test or overlap_val_test:
        raise AssertionError("Fuga de grupos: se detecto solapamiento entre subconjuntos.")

    repeated_groups = int(np.sum(group_sizes > 1))
    repeated_extra_rows = int(np.sum(group_sizes[group_sizes > 1] - 1))
    diagnostics = {
        "unique_feature_groups": int(n_groups),
        "repeated_feature_groups": repeated_groups,
        "extra_rows_in_repeated_groups": repeated_extra_rows,
        "overlap_train_val": overlap_train_val,
        "overlap_train_test": overlap_train_test,
        "overlap_val_test": overlap_val_test,
    }
    return train_idx, val_idx, test_idx, diagnostics


def standardize_fit(X_train):
    """Calcula media y desviacion solo con entrenamiento."""
    mean = np.mean(X_train, axis=0)
    std = np.std(X_train, axis=0, ddof=0)
    std[std == 0] = 1.0
    return mean, std


def standardize_transform(X, mean, std):
    return (np.asarray(X, dtype=float) - mean) / std


def knn_predict_one(X_train, y_train, x_new, k):
    """Predice por distancia euclidiana y votacion mayoritaria.

    En caso de empate se elige la etiqueta numericamente menor.
    """
    X_train = np.asarray(X_train, dtype=float)
    y_train = np.asarray(y_train)
    x_new = np.asarray(x_new, dtype=float)
    if X_train.ndim != 2 or x_new.ndim != 1:
        raise ValueError("Dimensiones incorrectas para X_train o x_new.")
    if X_train.shape[1] != x_new.shape[0]:
        raise ValueError("La muestra nueva debe tener el mismo numero de variables.")
    if len(X_train) != len(y_train):
        raise ValueError("X_train e y_train deben tener la misma cantidad de filas.")
    if not (1 <= k <= len(X_train)):
        raise ValueError("K debe estar entre 1 y la cantidad de muestras de entrenamiento.")

    distances = np.sqrt(np.sum((X_train - x_new) ** 2, axis=1))
    nearest_idx = np.argsort(distances, kind="stable")[:k]
    neighbor_labels = y_train[nearest_idx]
    labels, votes = np.unique(neighbor_labels, return_counts=True)
    max_votes = votes.max()
    prediction = np.min(labels[votes == max_votes])
    return prediction


def knn_predict(X_train, y_train, X_query, k):
    X_query = np.asarray(X_query, dtype=float)
    return np.array([knn_predict_one(X_train, y_train, row, k) for row in X_query])


def classification_metrics(y_true, y_pred):
    """Metricas para etiquetas binarias 0/1, calculadas sin librerias de ML."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    labels = [0, 1]
    cm = np.zeros((2, 2), dtype=int)
    for actual, predicted in zip(y_true, y_pred):
        if actual in labels and predicted in labels:
            cm[labels.index(actual), labels.index(predicted)] += 1

    accuracy = float(np.trace(cm) / cm.sum()) if cm.sum() else 0.0
    per_class_f1 = []
    per_class = {}
    for i, label in enumerate(labels):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (2 * precision * recall / (precision + recall)
              if (precision + recall) else 0.0)
        per_class[label] = {"precision": precision, "recall": recall, "f1": f1}
        per_class_f1.append(f1)

    return {
        "accuracy": accuracy,
        "macro_f1": float(np.mean(per_class_f1)),
        "confusion_matrix": cm,
        "per_class": per_class,
    }


def run_unit_tests():
    """Tres pruebas pequenas con resultados conocidos y calculables a mano."""
    X_demo = np.array([
        [0.0, 0.0], [1.0, 0.0], [0.0, 1.0], [5.0, 5.0], [6.0, 5.0]
    ])
    y_demo = np.array([0, 0, 0, 1, 1])

    p1 = knn_predict_one(X_demo, y_demo, np.array([0.2, 0.2]), k=1)
    assert p1 == 0, f"Prueba 1 fallo: se esperaba 0 y salio {p1}"

    p2 = knn_predict_one(X_demo, y_demo, np.array([5.5, 5.2]), k=3)
    assert p2 == 1, f"Prueba 2 fallo: se esperaba 1 y salio {p2}"

    X_tie = np.array([[0.0, 0.0], [2.0, 0.0]])
    y_tie = np.array([0, 1])
    p3 = knn_predict_one(X_tie, y_tie, np.array([1.0, 0.0]), k=2)
    assert p3 == 0, f"Prueba 3 fallo: se esperaba 0 por desempate y salio {p3}"

    print("\n=== PRUEBAS UNITARIAS DEL NUCLEO K-NN ===")
    print("Prueba 1: OK - K=1, prediccion esperada Clase 0")
    print("Prueba 2: OK - K=3, prediccion esperada Clase 1")
    print("Prueba 3: OK - empate, regla de desempate Clase 0")


def main():
    print("SIRCBF - AVANCE 2 CORREGIDO | K-NN implementado desde cero")
    df = load_uci_banknote()
    df = clean_and_report(df)
    create_eda(df)

    X = df[FEATURES].to_numpy(dtype=float)
    y = df["class"].to_numpy(dtype=int)
    train_idx, val_idx, test_idx, diagnostics = stratified_group_split_indices(X, y)

    X_train, y_train = X[train_idx], y[train_idx]
    X_val, y_val = X[val_idx], y[val_idx]

    print("\n=== DIVISION ESTRATIFICADA POR GRUPOS DE CARACTERISTICAS (SEMILLA 42) ===")
    print(f"Entrenamiento: {len(train_idx)} muestras")
    print(f"Validacion:    {len(val_idx)} muestras")
    print(f"Prueba:        {len(test_idx)} muestras (reservadas; no evaluadas ahora)")
    print(f"Grupos unicos de caracteristicas: {diagnostics['unique_feature_groups']}")
    print(f"Grupos con caracteristicas repetidas: {diagnostics['repeated_feature_groups']}")
    print(f"Filas adicionales dentro de grupos repetidos: {diagnostics['extra_rows_in_repeated_groups']}")
    print("Solapamiento de grupos train-validacion:", diagnostics["overlap_train_val"])
    print("Solapamiento de grupos train-prueba:", diagnostics["overlap_train_test"])
    print("Solapamiento de grupos validacion-prueba:", diagnostics["overlap_val_test"])
    print("Verificacion: ningun grupo de caracteristicas aparece en mas de un subconjunto.")

    # Reportar distribucion por clase en cada subconjunto.
    print("\nDistribucion de clases por subconjunto:")
    for name, labels in [("Entrenamiento", y_train), ("Validacion", y_val)]:
        counts = pd.Series(labels).value_counts().sort_index().to_dict()
        print(f"{name}: {counts}")
    print("Prueba: reservada; no se muestran sus etiquetas ni se calculan metricas en este avance.")

    majority_class = int(pd.Series(y_train).value_counts().idxmax())
    baseline_pred = np.full(shape=len(y_val), fill_value=majority_class, dtype=int)
    baseline_metrics = classification_metrics(y_val, baseline_pred)
    print("\n=== BASELINE: CLASE MAYORITARIA ===")
    print(f"Clase mayoritaria en entrenamiento: {majority_class}")
    print(f"Exactitud en validacion: {baseline_metrics['accuracy']:.4f}")
    print(f"F1 macro en validacion: {baseline_metrics['macro_f1']:.4f}")
    print("Matriz de confusion [filas=real 0/1, columnas=predicho 0/1]:")
    print(baseline_metrics["confusion_matrix"])

    # Ajustar la normalizacion solo con entrenamiento.
    mean, std = standardize_fit(X_train)
    X_train_scaled = standardize_transform(X_train, mean, std)
    X_val_scaled = standardize_transform(X_val, mean, std)

    print("\n=== SELECCION DE K EN VALIDACION ===")
    results = []
    for k in K_CANDIDATES:
        predictions = knn_predict(X_train_scaled, y_train, X_val_scaled, k)
        metrics = classification_metrics(y_val, predictions)
        results.append((k, metrics["accuracy"], metrics["macro_f1"], metrics))
        print(f"K={k:2d} | accuracy={metrics['accuracy']:.4f} | macro-F1={metrics['macro_f1']:.4f}")

    best_k, best_acc, best_macro_f1, best_metrics = sorted(
        results, key=lambda row: (-row[1], -row[2], row[0])
    )[0]
    print(f"\nK seleccionado con validacion: {best_k}")
    print(f"Exactitud K-NN en validacion: {best_acc:.4f}")
    print(f"F1 macro K-NN en validacion: {best_macro_f1:.4f}")
    print("Matriz de confusion K-NN [filas=real 0/1, columnas=predicho 0/1]:")
    print(best_metrics["confusion_matrix"])
    print("Comparacion: baseline accuracy = "
          f"{baseline_metrics['accuracy']:.4f}; K-NN accuracy = {best_acc:.4f}")
    print("NOTA: el conjunto de prueba permanece reservado para la entrega final.")

    run_unit_tests()

    # Guardar un resumen reproducible del avance, sin incluir metricas del test.
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    summary = {
        "seed": SEED,
        "samples_total": int(len(df)),
        "train_samples": int(len(train_idx)),
        "validation_samples": int(len(val_idx)),
        "test_samples_reserved": int(len(test_idx)),
        "duplicate_full_rows_preserved": int(df.duplicated(subset=FEATURES + ["class"]).sum()),
        "group_diagnostics": diagnostics,
        "baseline_validation_accuracy": baseline_metrics["accuracy"],
        "baseline_validation_macro_f1": baseline_metrics["macro_f1"],
        "selected_k": int(best_k),
        "knn_validation_accuracy": best_acc,
        "knn_validation_macro_f1": best_macro_f1,
        "test_evaluated": False,
    }
    with (OUT_DIR / "resumen_avance2.json").open("w", encoding="utf-8") as file:
        json.dump(summary, file, indent=2, ensure_ascii=False)

    print("\n=== FIN DEL AVANCE 2 CORREGIDO ===")
    print(f"Evidencias guardadas en: {OUT_DIR.resolve()}")
    print("Archivos: 01_distribuciones.png, 02_balance_clases.png, 03_correlaciones.png, resumen_avance2.json")
    print("El conjunto de prueba sigue reservado y no fue evaluado.")


if __name__ == "__main__":
    main()


## Evidencias que deben guardarse

1. Control de calidad: faltantes, infinitos y duplicados.
2. Estadísticas descriptivas, balance de clases y correlaciones.
3. Las tres gráficas generadas.
4. Diagnóstico de división agrupada y comprobación de solapamiento igual a cero.
5. Resultados del baseline y selección de K en validación.
6. Mensaje de aprobación de las tres pruebas unitarias.
7. Archivo `resumen_avance2.json`.
8. Fotografías o escaneo de las derivaciones y prueba de escritorio hechas a mano.

**Importante:** no copiar los resultados de la ejecución anterior. Registrar únicamente las cifras que aparezcan al ejecutar esta versión corregida. El conjunto de prueba no debe evaluarse en este avance.